# Điều tra dịch bệnh ở Vastopolis

*Bài thực hành khoa học dữ liệu: trực quan hóa và kể chuyện với dữ liệu.*

Bạn là nhà phân tích dữ liệu của sở y tế thành phố **Vastopolis** (khoảng 2 triệu dân). Sáng nay, bệnh viện báo có nhiều ca bệnh lạ.
Bạn chỉ có **hơn một triệu bài đăng mạng xã hội** có gắn vị trí GPS và thời gian, cùng bản đồ, dân số và thời tiết.

Ba câu hỏi cần trả lời:

1. Bệnh **lây lan bằng cách nào**?
2. Dịch đã **được kiểm soát** hay chưa?
3. Bệnh **bắt nguồn từ đâu**?

Quy tắc của một nhà điều tra giỏi: **mỗi biểu đồ chỉ trả lời một câu hỏi, mỗi kết quả dẫn đến một giả thuyết mới,
mỗi giả thuyết phải được kiểm tra bằng một loại dữ liệu khác.** Sổ tay này sẽ đi lần lượt qua bảy bước điều tra,
và **không có đáp án nào được nêu trước khi dữ liệu cho thấy nó**.

Dữ liệu: VAST Challenge 2011, Mini-Challenge 1 (đây là dữ liệu mô phỏng, thành phố Vastopolis là hư cấu).

## Bước 1. Hồ sơ vụ án: ta đang có gì trong tay?

Việc đầu tiên của người phân tích là **kiểm kê**: tải dữ liệu về, xem có những tệp nào, mỗi tệp lớn cỡ nào.

In [ ]:
import os, re, urllib.request, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from PIL import Image

URL = "https://visualdata.wustl.edu/varepository/VAST_Challenge_2011/datasets/MC_1_Materials_3-30-2011.zip"
FOLDER = "MC_1_Materials_3-30-2011"

if not os.path.isdir(FOLDER):              # chỉ tải khi chưa có
    urllib.request.urlretrieve(URL, "mc1.zip")
    zipfile.ZipFile("mc1.zip").extractall(".")

for name in sorted(os.listdir(FOLDER)):
    size = os.path.getsize(os.path.join(FOLDER, name)) / 1e6
    print(f"{name:26s} {size:8.2f} MB")

In [ ]:
# Định dạng đồ thị dùng chung (bảng màu dễ nhìn, phông có dấu tiếng Việt)
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "axes.titleweight": "bold",
    "axes.titlesize": 12, "axes.labelsize": 10, "font.size": 10,
})
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

FLU, GI, EVT = "#E4572E", "#0077B6", "#7B2CBF"     # cam đỏ, xanh nước, tím
NAVY, GRAY, GOLD = "#1F2A44", "#9AA5B1", "#F2A900"


def ngay_thang(ax, dinh_dang="%d/%m"):
    ax.xaxis.set_major_formatter(mdates.DateFormatter(dinh_dang))


def base_map(ax, alpha=0.45):
    """Vẽ ảnh bản đồ Vastopolis làm nền (nhạt) với trục kinh độ, vĩ độ."""
    ax.imshow(MAP_SMALL, extent=EXTENT, alpha=alpha, aspect=1 / np.cos(np.radians(42.2)), zorder=0)
    ax.set_xlim(EXTENT[0], EXTENT[1]); ax.set_ylim(EXTENT[2], EXTENT[3])
    ax.set_xlabel("kinh độ"); ax.set_ylabel("vĩ độ"); ax.grid(False)


def table_png(frame, title, widths=None):
    """Vẽ một bảng dữ liệu thành hình ảnh."""
    fig, ax = plt.subplots(figsize=(11, 0.4 * len(frame) + 1.2))
    ax.axis("off")
    tb = ax.table(cellText=frame.values, colLabels=list(frame.columns), cellLoc="left",
                  colWidths=widths, loc="center")
    tb.auto_set_font_size(False); tb.set_fontsize(8.5); tb.scale(1, 1.5)
    for (r, c), cell in tb.get_celld().items():
        cell.set_edgecolor("#D5DAE1")
        if r == 0:
            cell.set_facecolor(NAVY); cell.set_text_props(color="white", weight="bold")
        elif r % 2 == 0:
            cell.set_facecolor("#F3F5F8")
    ax.set_title(title, loc="left")
    plt.tight_layout()

Có **năm tệp**. Tệp lớn nhất là `Microblogs.csv` (hơn 100 MB): đó là các bài đăng. Hãy đọc nó và xem **cấu trúc** (số dòng, số cột, kiểu dữ liệu).

In [ ]:
df = pd.read_csv(f"{FOLDER}/Microblogs.csv", encoding="latin-1")
print("Số dòng, số cột:", df.shape)
print()
print(df.dtypes)
print()
print(df.head(5).to_string(max_colwidth=48))

In [ ]:
mau = df.sample(8, random_state=7)[["ID", "Created_at", "Location", "text"]].copy()
mau["text"] = mau["text"].fillna("").str.slice(0, 78)
table_png(mau, "Tám dòng ngẫu nhiên của Microblogs.csv", widths=[0.06, 0.13, 0.17, 0.64])
plt.show()

Mỗi dòng là **một bài đăng**: ai đăng (`ID`), đăng lúc nào (`Created_at`), đăng ở đâu (`Location`: vĩ độ và kinh độ) và nội dung (`text`).
Ngoài ra còn hai bảng nhỏ về dân số và thời tiết.

In [ ]:
pop = pd.read_csv(f"{FOLDER}/Population.csv")
weather = pd.read_csv(f"{FOLDER}/Weather.csv")
print("Population.csv:", pop.shape)
print(pop.head(6).to_string(index=False))
print()
print("Weather.csv:", weather.shape)
print(weather.head(6).to_string(index=False))

In [ ]:
MAP = Image.open(f"{FOLDER}/Vastopolis_Map.png")
print("Ảnh bản đồ:", MAP.size, "điểm ảnh, chế độ", MAP.mode)
MAP_SMALL = MAP.convert("RGB").resize((2600, 1322))      # thu nhỏ để vẽ nhanh

fig, ax = plt.subplots(figsize=(10, 5.1))
ax.imshow(MAP_SMALL); ax.axis("off")
ax.set_title("Vastopolis_Map.png: 13 khu vực, đường cao tốc, bệnh viện, sông")
plt.tight_layout()
plt.show()

## Bước 2. Kiểm tra chất lượng: dữ liệu có đáng tin không?

Một nhà điều tra không tin lời khai khi chưa kiểm tra. Ta kiểm tra ba thứ: **vị trí**, **thời gian**, **nội dung**.
Cột `Location` đang là chuỗi `"vĩ độ kinh độ"`; cần tách thành hai số và đổi cột thời gian sang kiểu ngày giờ.

In [ ]:
loc = df["Location"].str.split(" ", expand=True).astype(float)
print("Kinh độ thô: nhỏ nhất", loc[1].min(), "| lớn nhất", loc[1].max())

df["lat"] = loc[0]
df["lon"] = -loc[1]            # tệp ghi kinh độ Tây mà thiếu dấu trừ, ta thêm lại
df["t"] = pd.to_datetime(df["Created_at"], format="%m/%d/%Y %H:%M", errors="coerce")

bad = df[df["t"].isna()]
print("Dòng có thời gian hỏng:", len(bad))
print(bad[["ID", "Created_at"]].head(4).to_string(index=False))
print("Bài không có nội dung:", df["text"].isna().sum(), "| dòng trùng hệt:", df.duplicated().sum())
print(df[["lat", "lon"]].describe().round(4).to_string())

Hai điều cần ghi nhớ: (1) **kinh độ bị thiếu dấu trừ**, và (2) **một số ít dòng có thời gian hỏng**.
Đừng lật dấu một cách mù quáng: hãy vẽ thử các bài đăng lên bản đồ để xem phép chiếu có hợp lý không.
Giả định: ảnh bản đồ phủ vừa khít vùng dữ liệu (góc ảnh trùng giá trị nhỏ nhất và lớn nhất của tọa độ).

In [ ]:
EXTENT = [df["lon"].min(), df["lon"].max(), df["lat"].min(), df["lat"].max()]

fig, ax = plt.subplots(figsize=(10, 5.1))
base_map(ax)
s = df.sample(40000, random_state=1)
ax.scatter(s["lon"], s["lat"], s=1.2, c=NAVY, alpha=0.35)
ax.set_title("40 000 bài đăng ngẫu nhiên đặt lên bản đồ")
plt.tight_layout()
plt.show()

In [ ]:
df = df.dropna(subset=["t"]).copy()               # bỏ dòng thời gian hỏng
df["text"] = df["text"].fillna("")
df["tl"] = df["text"].str.lower()                  # bản chữ thường để tìm từ khóa

per_user = df.groupby("ID").size()
print("Sau khi làm sạch:", len(df), "bài của", df["ID"].nunique(), "người dùng")
print("Khoảng thời gian:", df["t"].min(), "->", df["t"].max())
print("Số bài mỗi người:")
print(per_user.describe().round(2).to_string())

## Bước 3. Dòng thời gian: khi nào có điều bất thường?

Câu hỏi đầu tiên của thám tử luôn là **"khi nào?"**. Đếm số bài đăng mỗi ngày, rồi so với mức bình thường của 14 ngày đầu.

In [ ]:
daily = df.groupby(df["t"].dt.date).size()
base = daily.iloc[:14]                              # 14 ngày đầu làm mốc bình thường
z = (daily - base.mean()) / base.std()

print(f"Mức bình thường: {base.mean():,.0f} bài/ngày (độ lệch chuẩn {base.std():,.0f})")
print(pd.DataFrame({"bài": daily, "z": z.round(1)}).tail(7).to_string())

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(daily.index, daily.values, color=NAVY, lw=2, marker="o", ms=4)
hot = daily[z > 3]
ax.scatter(hot.index, hot.values, color=FLU, s=60, zorder=3, label="vượt mốc 3 độ lệch chuẩn")
ax.axhline(base.mean(), color=GRAY, ls="--", lw=1)
ax.set_title("Số bài đăng mỗi ngày"); ax.set_ylabel("số bài"); ax.legend(frameon=False)
ngay_thang(ax); plt.tight_layout()
plt.show()

In [ ]:
tab = df.groupby(df["t"].dt.floor("h")).size().rename("n").to_frame()
tab["gio"] = tab.index.hour
tab["ngay"] = tab.index.date
binh_thuong = tab.iloc[: 14 * 24].groupby("gio")["n"].mean()       # mức theo từng giờ trong ngày
tab["ti_le"] = tab["n"] / tab["gio"].map(binh_thuong)
ratio = tab.pivot(index="ngay", columns="gio", values="ti_le")     # hàng: ngày, cột: giờ

print("Lần đầu lượng bài vượt 1,5 lần bình thường:", tab[tab["ti_le"] > 1.5].index[0])

fig, ax = plt.subplots(figsize=(10, 4.6))
im = ax.imshow(ratio.values, aspect="auto", cmap="YlOrRd", vmin=0.8, vmax=2.2)
ax.set_xticks(range(0, 24, 3)); ax.set_xticklabels([f"{h}h" for h in range(0, 24, 3)])
ax.set_yticks(range(len(ratio))); ax.set_yticklabels([d.strftime("%d/%m") for d in ratio.index], fontsize=8)
ax.set_title("Số bài mỗi giờ so với mức bình thường của giờ đó (lần)"); ax.grid(False)
plt.colorbar(im, ax=ax, label="lần")
plt.tight_layout()
plt.show()

## Bước 4. Người ta đang nói về điều gì?

Số bài tăng, nhưng **tăng vì chủ đề nào**? Không đoán từ khóa: để dữ liệu tự nói.
Với mỗi từ, ta tính **độ nổi bật (lift)** = tần suất của từ trong ngày đó chia cho tần suất của từ trong những ngày bình thường.
Từ nào có lift rất lớn là từ "bỗng dưng được nhắc nhiều".

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from scipy import sparse

vec = CountVectorizer(token_pattern=r"[a-z']{3,}", min_df=30, binary=True)
X = vec.fit_transform(df["tl"])                        # bài × từ (có hoặc không)
vocab = np.array(vec.get_feature_names_out())

ngay0 = df["t"].dt.normalize().min()
day_idx = (df["t"].dt.normalize() - ngay0).dt.days.values
D = sparse.csr_matrix((np.ones(len(df)), (np.arange(len(df)), day_idx)))
C = (X.T @ D).toarray()                                # từ × ngày: số bài chứa từ
N = np.bincount(day_idx)                               # số bài mỗi ngày


def top_words(ngay, k=10, toi_thieu=300):
    """Từ nổi bật nhất của một ngày so với TẤT CẢ các ngày trước đó."""
    i = (pd.Timestamp(ngay) - ngay0).days
    truoc = C[:, :i].sum(1) / N[:i].sum()               # tần suất trước ngày đó
    lift = (C[:, i] / N[i]) / (truoc + 1 / N.sum())
    lift[C[:, i] < toi_thieu] = 0                       # bỏ từ quá hiếm (lỗi chính tả)
    best = np.argsort(-lift)[:k]
    return [(vocab[j], int(C[j, i]), round(float(lift[j]))) for j in best]


for ngay in ["2011-05-18", "2011-05-19", "2011-05-20"]:
    print(ngay[5:], [f"{w} ({lift}x)" for w, c, lift in top_words(ngay)])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax, ngay, col in zip(axes, ["2011-05-18", "2011-05-19", "2011-05-20"], [FLU, GI, NAVY]):
    tw = top_words(ngay, k=8)[::-1]
    ax.barh([w for w, c, l in tw], [l for w, c, l in tw], color=col)
    ax.set_xscale("log"); ax.set_title(ngay[8:] + "/05")
    ax.tick_params(axis="x", which="minor", labelbottom=False)
    ax.set_xlabel("lift (lần, thang log)")
plt.tight_layout()
plt.show()

Các từ nổi bật là **triệu chứng**: sốt, ớn lạnh, khó thở, mệt mỏi... rồi sang ngày sau là buồn nôn, tiêu chảy, đau bụng.
Hai nhóm triệu chứng khác nhau. Ta gom từ khóa thành hai nhóm rồi đếm theo ngày:
nhóm **giống cúm** (hô hấp) và nhóm **dạ dày - ruột**.

In [ ]:
FLU_WORDS = ["fever", "chills", "sweats", "fatigue", "cough", "coughing", "shortness", "breath",
             "headache", "ache", "aches", "aching", "runny", "sore throat", "pneumonia", "flu",
             "bedridden", "debilitated", "sickly", "flem", "phlegm"]
GI_WORDS = ["diarrhea", "nausea", "nauseous", "vomit", "vomiting", "vomitting", "barfed", "puking",
            "cramps", "stomach", "heartburn", "tummy", "toilet", "queasy"]


def pattern(words):
    return r"\b(?:" + "|".join(words) + r")\b"


df["flu"] = df["tl"].str.contains(pattern(FLU_WORDS))
df["gi"] = df["tl"].str.contains(pattern(GI_WORDS))
df["ngay"] = df["t"].dt.date

theo_ngay = df.groupby("ngay")[["flu", "gi"]].sum()
print("Trung bình 14 ngày đầu:", theo_ngay.iloc[:14].mean().round(0).to_dict())
print(theo_ngay.tail(5).to_string())

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(theo_ngay.index, theo_ngay["flu"], color=FLU, lw=2.4, marker="o", ms=4, label="giống cúm")
ax.plot(theo_ngay.index, theo_ngay["gi"], color=GI, lw=2.4, marker="o", ms=4, label="dạ dày - ruột")
ax.set_title("Số bài nhắc đến từng nhóm triệu chứng mỗi ngày"); ax.set_ylabel("số bài")
ax.legend(frameon=False); ngay_thang(ax); plt.tight_layout()
plt.show()

In [ ]:
def thoi_diem_bat_dau(cot, so_lan=5):
    gio = df.groupby(df["t"].dt.floor("h"))[cot].sum()
    nen_gio = gio.iloc[: 14 * 24]
    nguong = nen_gio.mean() + so_lan * nen_gio.std()
    vuot = gio[gio > nguong]
    return vuot.index[0], round(nguong)


for cot, ten in [("flu", "giống cúm"), ("gi", "dạ dày - ruột")]:
    t0, nguong = thoi_diem_bat_dau(cot)
    print(f"Nhóm {ten}: vượt ngưỡng {nguong} bài/giờ lần đầu lúc {t0}")

gio = df.groupby(df["t"].dt.floor("h"))[["flu", "gi"]].sum()
gio = gio["2011-05-17":"2011-05-20"]
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.fill_between(gio.index, gio["flu"], color=FLU, alpha=0.35); ax.plot(gio.index, gio["flu"], color=FLU, label="giống cúm")
ax.fill_between(gio.index, gio["gi"], color=GI, alpha=0.35); ax.plot(gio.index, gio["gi"], color=GI, label="dạ dày - ruột")
ax.set_title("Bài đăng triệu chứng theo giờ, 17 đến 20/5"); ax.set_ylabel("bài mỗi giờ")
ax.legend(frameon=False); ngay_thang(ax, "%d/%m %Hh"); plt.tight_layout()
plt.show()

## Bước 5. Khoanh vùng: các ca bệnh ở đâu?

Biết **khi nào** rồi, giờ hỏi **ở đâu**. Vẽ riêng hai nhóm triệu chứng của ngày 19/5 lên bản đồ.
Nếu cả hai nhóm cùng một bệnh, ta mong thấy chúng chồng lên nhau. Có đúng vậy không?

In [ ]:
from datetime import date

ngay19 = df[df["ngay"] == date(2011, 5, 19)]
ten = {"flu": "giống cúm", "gi": "dạ dày - ruột"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
for ax, cot, mau in zip(axes, ["flu", "gi"], [FLU, GI]):
    d = ngay19[ngay19[cot]]
    base_map(ax, 0.35)
    ax.scatter(d["lon"], d["lat"], s=1.5, c=mau, alpha=0.4)
    ax.set_title(f"{ten[cot]}: {len(d):,} bài ngày 19/5")
    print(ten[cot], len(d), "bài")
plt.tight_layout()
plt.show()

In [ ]:
from scipy.ndimage import gaussian_filter

xb = np.linspace(EXTENT[0], EXTENT[1], 140)
yb = np.linspace(EXTENT[2], EXTENT[3], 70)


def luoi(d):
    """Đếm số bài trong từng ô lưới."""
    H, _, _ = np.histogram2d(d["lat"], d["lon"], bins=[yb, xb])
    return H


nen = df[df["ngay"] < date(2011, 5, 15)]
n_nen = nen["ngay"].nunique()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
for ax, cot, cmap in zip(axes, ["flu", "gi"], ["Oranges", "Blues"]):
    vuot = luoi(ngay19[ngay19[cot]]) - luoi(nen[nen[cot]]) / n_nen     # trừ mức bình thường mỗi ngày
    vuot = gaussian_filter(np.clip(vuot, 0, None), 1.2)
    iy, ix = np.unravel_index(vuot.argmax(), vuot.shape)
    print(f"{ten[cot]}: ô đậm nhất quanh kinh độ {xb[ix]:.3f}, vĩ độ {yb[iy]:.3f}")
    base_map(ax, 0.3)
    ax.imshow(np.ma.masked_less(vuot, vuot.max() * 0.04), extent=EXTENT, origin="lower", cmap=cmap,
              aspect=1 / np.cos(np.radians(42.2)), alpha=0.85, zorder=2)
    ax.set_title(f"Mật độ bài {ten[cot]} vượt mức bình thường")
plt.tight_layout()
plt.show()

Hai nhóm **không chồng lên nhau**: một nhóm tập trung ở trung tâm thành phố, nhóm kia kéo dài về phía tây nam.
Nhưng thành phố có khu rất đông, có khu rất thưa. Số bài thô dễ đánh lừa, nên ta **chuẩn hóa theo dân số**:
số bài triệu chứng trên **1000 cư dân** của từng khu.
Bản đồ chỉ cho ảnh chứ không có tệp ranh giới, nên ta ước lượng ranh giới 13 khu bằng nhiều điểm mốc đọc từ ảnh.

In [ ]:
ZONES = {   # vài điểm mốc (kinh độ, vĩ độ) bên trong mỗi khu, đọc từ ảnh bản đồ
    "Cornertown": [(-93.55, 42.295), (-93.53, 42.292), (-93.515, 42.295), (-93.52, 42.285)],
    "Northville": [(-93.47, 42.29), (-93.44, 42.295), (-93.45, 42.275), (-93.435, 42.248), (-93.455, 42.262)],
    "Riverside": [(-93.395, 42.285), (-93.37, 42.29), (-93.40, 42.26), (-93.375, 42.265), (-93.36, 42.28)],
    "Westside": [(-93.50, 42.255), (-93.485, 42.235), (-93.52, 42.235), (-93.51, 42.215), (-93.485, 42.215),
                 (-93.465, 42.235), (-93.47, 42.255)],
    "Villa": [(-93.555, 42.26), (-93.555, 42.23), (-93.55, 42.205), (-93.545, 42.215), (-93.55, 42.24)],
    "Smogtown": [(-93.55, 42.175), (-93.53, 42.165), (-93.54, 42.185), (-93.56, 42.17)],
    "Plainville": [(-93.47, 42.18), (-93.45, 42.195), (-93.49, 42.17), (-93.43, 42.2), (-93.44, 42.18),
                   (-93.50, 42.185), (-93.42, 42.208)],
    "Southville": [(-93.40, 42.172), (-93.42, 42.19), (-93.39, 42.165), (-93.38, 42.19), (-93.43, 42.17)],
    "Downtown": [(-93.38, 42.218), (-93.36, 42.21), (-93.40, 42.212), (-93.34, 42.225), (-93.375, 42.228),
                 (-93.39, 42.222)],
    "Uptown": [(-93.345, 42.24), (-93.33, 42.245), (-93.36, 42.243), (-93.35, 42.232)],
    "Lakeside": [(-93.30, 42.18), (-93.25, 42.19), (-93.22, 42.17), (-93.30, 42.165), (-93.32, 42.20),
                 (-93.345, 42.185), (-93.26, 42.205), (-93.22, 42.20), (-93.30, 42.198)],
    "Eastside": [(-93.28, 42.225), (-93.23, 42.225), (-93.21, 42.23), (-93.30, 42.232), (-93.25, 42.218),
                 (-93.31, 42.222), (-93.22, 42.218)],
    "Suburbia": [(-93.27, 42.275), (-93.22, 42.27), (-93.25, 42.25), (-93.30, 42.29), (-93.22, 42.30),
                 (-93.32, 42.265), (-93.28, 42.245), (-93.325, 42.29), (-93.22, 42.248), (-93.30, 42.248)],
}
ten_khu = list(ZONES)
mot = np.array([p for k in ten_khu for p in ZONES[k]])
nhan = np.array([i for i, k in enumerate(ten_khu) for _ in ZONES[k]])

co = np.cos(np.radians(42.2))                      # co dãn kinh độ ở vĩ độ 42 độ
gx = np.linspace(EXTENT[0], EXTENT[1], 700)
gy = np.linspace(EXTENT[3], EXTENT[2], 350)
GX, GY = np.meshgrid(gx, gy)
gan_nhat = ((GX[..., None] - mot[:, 0]) * co) ** 2 + (GY[..., None] - mot[:, 1]) ** 2
RASTER = nhan[gan_nhat.argmin(-1)]                 # mỗi ô lưới thuộc khu có điểm mốc gần nhất


def khu_cua(lon, lat):
    ix = np.clip(((lon - EXTENT[0]) / (EXTENT[1] - EXTENT[0]) * 699).astype(int), 0, 699)
    iy = np.clip(((EXTENT[3] - lat) / (EXTENT[3] - EXTENT[2]) * 349).astype(int), 0, 349)
    return RASTER[iy, ix]


df["khu"] = khu_cua(df["lon"].values, df["lat"].values)
dan = pop.set_index("Zone_Name")["Population_Density"].reindex(ten_khu).values

trieu_chung = df["flu"] | df["gi"]
d19 = df[(df["ngay"] == date(2011, 5, 19)) & trieu_chung]
d_nen = df[(df["ngay"] < date(2011, 5, 15)) & trieu_chung]
so_19 = d19.groupby("khu").size().reindex(range(13), fill_value=0).values
so_nen = d_nen.groupby("khu").size().reindex(range(13), fill_value=0).values / d_nen["ngay"].nunique()

ban_ngay = pop.set_index("Zone_Name")["Daytime_Population"].reindex(ten_khu).values
bang = pd.DataFrame({"khu": ten_khu, "dân": dan, "bài 19/5": so_19, "bình thường": so_nen.round(0),
                     "trên 1000 dân": (so_19 / dan * 1000).round(1),
                     "trên 1000 (ban ngày)": (so_19 / ban_ngay * 1000).round(1)})
print(bang.sort_values("trên 1000 dân", ascending=False).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
for ax, gia_tri, cmap, tieu_de in zip(axes, [so_19, so_19 / dan * 1000], ["YlOrBr", "YlOrRd"],
                                      ["Số bài triệu chứng thô", "Số bài trên 1000 cư dân"]):
    base_map(ax, 0.15)
    im = ax.imshow(gia_tri[RASTER], extent=EXTENT, cmap=cmap, alpha=0.75, aspect=1 / co, zorder=2)
    plt.colorbar(im, ax=ax, shrink=0.8)
    for k, ten_k in enumerate(ten_khu):
        x, y = np.mean(ZONES[ten_k], axis=0)
        ax.text(x, y, ten_k, fontsize=6.5, ha="center", color=NAVY, zorder=3)
    ax.set_title(tieu_de + " (19/5)")
plt.tight_layout()
plt.show()

## Bước 6. Nguồn bệnh: có chuyện gì xảy ra trước đó?

Bệnh không tự nhiên xuất hiện. Thám tử luôn hỏi: **trước ngày bệnh bùng phát, thành phố đã xảy ra sự kiện gì?**
Ta không đoán mà tìm các **từ bùng nổ**: từ nào bỗng được nhắc rất nhiều trong đúng một ngày rồi biến mất.

In [ ]:
trung_vi = np.median(C, axis=1)                        # số bài chứa từ trong một ngày điển hình
dinh, ngay_dinh = C.max(1), C.argmax(1)               # ngày cao nhất của từng từ
lan = dinh / (trung_vi + 3)
chon = (dinh >= 150) & (lan >= 8)

bung_no = pd.DataFrame({"tu": vocab[chon], "ngay": ngay_dinh[chon], "lan": lan[chon]})
for i, g in bung_no.sort_values("lan", ascending=False).groupby("ngay"):
    ngay_str = (ngay0 + pd.Timedelta(days=int(i))).strftime("%d/%m")
    print(ngay_str, "|", ", ".join(g["tu"].head(6)))

Có **nhiều ngày** có chuyện: một triển lãm, một lễ hội, một hội nghị, một vụ máy bay, một lời dọa đánh bom...
Ngày 17/5 có **hai** nhóm từ cùng lúc. Chưa thể kết luận gì. Việc cần làm: với mỗi sự kiện, tìm **vị trí**
(tâm của các bài đăng nhắc đến nó) và **độ tản** của chúng.
Tên sự kiện do ta đặt theo từ khóa nổi bật.

In [ ]:
SU_KIEN = {   # tên đặt theo từ khóa nổi bật, mẫu tìm kiếm, ngày
    "Triển lãm đồ cổ": (r"\bantique\b", "2011-05-04"),
    "Lễ hội": (r"\bfestival\b", "2011-05-07"),
    "Hội nghị": (r"\bconvention\b", "2011-05-11"),
    "Máy bay rơi": (r"\b(?:plane|airport|crashed|crashing)\b", "2011-05-13"),
    "Tòa nhà Capital": (r"\b(?:capital|blaze)\b", "2011-05-14"),
    "Dọa đánh bom": (r"\b(?:bomb|bombs|threats?)\b", "2011-05-16"),
    "Xe tải, tai nạn": (r"\b(?:truck|trucks|accident|wrecked|bridge)\b", "2011-05-17"),
    "Nổ, khói bụi": (r"\b(?:explosions?|smog|rattle|shook|boom|blast)\b", "2011-05-17"),
}
kx, ky = 69.17 * np.cos(np.radians(42.2)), 69.0          # số dặm ứng với 1 độ kinh độ, vĩ độ

hang, TAM, BAI_SK = [], {}, {}
for ten_sk, (mau_tu, ngay) in SU_KIEN.items():
    d = df[(df["ngay"] == pd.Timestamp(ngay).date()) & df["tl"].str.contains(mau_tu)]
    BAI_SK[ten_sk] = d
    tam = (d["lon"].median(), d["lat"].median())
    ban_kinh = np.median(np.hypot((d["lon"] - tam[0]) * kx, (d["lat"] - tam[1]) * ky))
    TAM[ten_sk] = tam
    hang.append([ten_sk, ngay[5:], len(d), round(tam[1], 4), round(tam[0], 4), round(ban_kinh, 2),
                 d["t"].quantile(0.1).strftime("%H:%M")])
bang_sk = pd.DataFrame(hang, columns=["sự kiện", "ngày", "số bài", "vĩ độ", "kinh độ", "tản (dặm)", "10% bài đầu"])
print(bang_sk.to_string(index=False))

In [ ]:
mau_sk = ["#8D99AE", "#2A9D8F", "#6A994E", "#264653", "#E07A5F", "#B5838D", EVT, "#F2A900"]
lech = {"Triển lãm đồ cổ": (-40, 32), "Lễ hội": (12, -14), "Hội nghị": (14, 16), "Dọa đánh bom": (-120, -30),
        "Tòa nhà Capital": (-50, 22)}                  # độ lệch nhãn để không đè lên nhau
fig, ax = plt.subplots(figsize=(10, 5.2))
base_map(ax, 0.4)
for (ten_sk, (mau_tu, ngay)), mau in zip(SU_KIEN.items(), mau_sk):
    d = BAI_SK[ten_sk]
    ax.scatter(d["lon"], d["lat"], s=3, color=mau, alpha=0.3)
    x, y = TAM[ten_sk]
    ax.scatter([x], [y], s=150, color=mau, edgecolor="white", linewidth=1.6, zorder=4)
    ax.annotate(f"{ten_sk} ({ngay[8:]}/5)", (x, y), xytext=lech.get(ten_sk, (8, 7)), textcoords="offset points",
                fontsize=8, weight="bold", color=NAVY, zorder=5,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.8))
ax.set_title("Tám sự kiện bất thường trước 18/5, mỗi sự kiện một chỗ")
plt.tight_layout()
plt.show()

Tám sự kiện ở bảy địa điểm khác nhau. **Ai là thủ phạm?** Cần một dữ kiện độc lập để lọc: **gió**.
Nếu độc tố bay trong không khí thì nó phải đi theo hướng gió. Xem bảng thời tiết.

In [ ]:
HUONG = {"N": 0, "NNE": 22.5, "NE": 45, "ENE": 67.5, "E": 90, "ESE": 112.5, "SE": 135, "SSE": 157.5,
         "S": 180, "SSW": 202.5, "SW": 225, "WSW": 247.5, "W": 270, "WNW": 292.5, "NW": 315, "NNW": 337.5}
weather["ngay"] = pd.to_datetime(weather["Date"], format="%m/%d/%Y").dt.date
weather["gio_tu"] = weather["Wind_Direction"].map(HUONG)         # gió thổi TỪ hướng này
weather["gio_den"] = (weather["gio_tu"] + 180) % 360             # nên thổi ĐẾN hướng ngược lại
w = weather[weather["ngay"] >= date(2011, 5, 14)].reset_index(drop=True)
print(w[["Date", "Weather", "Average_Wind_Speed", "Wind_Direction", "gio_den"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 2.9))
rad = np.radians(w["gio_den"])
mau = [GOLD if d in (date(2011, 5, 17), date(2011, 5, 18), date(2011, 5, 19)) else GRAY for d in w["ngay"]]
ax.quiver(range(len(w)), np.zeros(len(w)), np.sin(rad), np.cos(rad), color=mau, scale=9, width=0.012)
for i, r in w.iterrows():
    ax.text(i, -1.15, f"{r['ngay'].strftime('%d/%m')}\n{r['Average_Wind_Speed']} mph\ntừ {r['Wind_Direction']}",
            ha="center", va="top", fontsize=8)
ax.set_xlim(-0.6, len(w) - 0.4); ax.set_ylim(-2.3, 1); ax.axis("off")
ax.set_title("Hướng gió thổi đến (mũi tên hướng lên là về phía Bắc)")
plt.tight_layout()
plt.show()

Gió đổi hướng liên tục, nhưng từ 16 đến 19/5 nó ổn định, **thổi từ Tây sang Đông**.
Giả thuyết cần kiểm tra: *nếu nguồn bệnh giống cúm là một sự kiện ở đâu đó, thì các ca của sáng 18/5 phải nằm
**xuôi gió** của nguồn đó, trong một dải hẹp dọc theo hướng gió, và hầu như không có ca nào ở **ngược gió**.*
Ta kiểm tra từng nghi phạm với các bài giống cúm từ 6h đến 14h ngày 18/5.

In [ ]:
huong = weather.set_index("ngay").loc[date(2011, 5, 18), "gio_den"]
cua_so = df[(df["t"] >= "2011-05-18 06:00") & (df["t"] < "2011-05-18 14:00") & df["flu"]]
print(f"Gió 18/5 thổi về hướng {huong:.0f} độ (90 = Đông). Số bài giống cúm trong cửa sổ: {len(cua_so)}")


def toa_do_gio(d, tam, huong):
    """Khoảng cách dọc theo gió và lệch ngang so với đường gió đi qua `tam` (đơn vị dặm)."""
    dx, dy = (d["lon"] - tam[0]) * kx, (d["lat"] - tam[1]) * ky
    a = np.radians(huong)
    return (dx * np.sin(a) + dy * np.cos(a)).values, (-dx * np.cos(a) + dy * np.sin(a)).values


hang = []
for ten_sk, tam in TAM.items():
    doc, ngang = toa_do_gio(cua_so, tam, huong)
    hang.append([ten_sk, 100 * (doc < 0).mean(), 100 * ((doc > 0) & (np.abs(ngang) < 1.5)).mean()])
diem = pd.DataFrame(hang, columns=["sự kiện", "% ca ngược gió", "% ca trong dải xuôi gió"]).set_index("sự kiện")
print(diem.round(1).sort_values("% ca trong dải xuôi gió", ascending=False).to_string())

fig, ax = plt.subplots(figsize=(9, 3.6))
s = diem["% ca trong dải xuôi gió"].sort_values()
ax.barh(s.index, s.values, color=[FLU if v == s.max() else GRAY for v in s.values])
ax.set_xlabel("% số ca giống cúm (6h-14h, 18/5) nằm trong dải hẹp xuôi gió của sự kiện")
ax.set_title("Sự kiện nào giải thích được vị trí các ca giống cúm?")
plt.tight_layout()
plt.show()

Đã có một nghi phạm dẫn đầu. Trước khi đi tiếp, quay lại **hiện trường**: những người dân thật sự viết gì
về sự kiện đó trong ngày 17/5, và con số có lớn bất thường so với ngày thường không?

In [ ]:
co_hai = df["tl"].str.contains(r"\btrucks?\b") & df["tl"].str.contains(r"\baccident\b")
so_ngay = co_hai.groupby(df["ngay"]).sum()
print(f"Bài chứa cả 'truck' lẫn 'accident': ngày 17/5 có {so_ngay[date(2011, 5, 17)]}, "
      f"trung bình mỗi ngày trong 14 ngày đầu chỉ {so_ngay.iloc[:14].mean():.1f}")

hien_truong = df[co_hai & (df["ngay"] == date(2011, 5, 17))]
print("Giờ đăng sớm của nhóm bài này (10% đầu tiên):", hien_truong["t"].quantile(0.1).strftime("%H:%M"))
print("Ba bài đăng thật:")
for noi_dung in hien_truong["text"].sample(3, random_state=4):
    print(" -", noi_dung[:100])

Có **một** nghi phạm khớp với hướng gió nhất: khoảng 90% ca giống cúm nằm trong một dải hẹp xuôi gió của nó,
và rất ít ca ở ngược gió (một nghi phạm khác cũng đạt mức khá cao, nên ta vẫn phải thử bằng dữ liệu khác). Ta tạm gọi đó là **nghi phạm số một** (một *giả thuyết*, chưa phải kết luận).
Giả thuyết cần được thử thách bằng dữ liệu khác, và còn một nhóm triệu chứng chưa giải thích: nhóm dạ dày - ruột.

## Bước 7. Lan truyền: bệnh đi như thế nào?

Trước hết, quan sát **sự lan rộng theo thời gian**: các ca giống cúm, mỗi khung 6 giờ, từ 18/5 đến 20/5.
Ngôi sao vàng là nghi phạm số một.

In [ ]:
NGUON = diem["% ca trong dải xuôi gió"].idxmax()           # nghi phạm số một theo kiểm tra trước
print("Nghi phạm số một:", NGUON, "tại", np.round(TAM[NGUON], 4))

moc = pd.date_range("2011-05-18", periods=12, freq="6h")
fig, axes = plt.subplots(3, 4, figsize=(12, 6.6))
for ax, t0 in zip(axes.ravel(), moc):
    d = df[(df["t"] >= t0) & (df["t"] < t0 + pd.Timedelta(hours=6)) & df["flu"]]
    base_map(ax, 0.25)
    ax.scatter(d["lon"], d["lat"], s=0.6, c=FLU, alpha=0.35)
    ax.plot(*TAM[NGUON], marker="*", ms=11, mfc=GOLD, mec=NAVY, zorder=5)
    ax.set_title(t0.strftime("%d/%m %Hh") + f": {len(d):,} bài", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image as PILImage

khung = []
moc3 = pd.date_range("2011-05-18", periods=24, freq="3h")
for t0 in moc3:
    d = df[(df["t"] >= t0) & (df["t"] < t0 + pd.Timedelta(hours=3)) & df["flu"]]
    fig, ax = plt.subplots(figsize=(6.4, 3.4), dpi=80)
    base_map(ax, 0.3)
    ax.scatter(d["lon"], d["lat"], s=1.2, c=FLU, alpha=0.4)
    ax.plot(*TAM[NGUON], marker="*", ms=13, mfc=GOLD, mec=NAVY, zorder=5)
    ax.set_title(f"Ca giống cúm, {t0:%d/%m %Hh} đến {t0 + pd.Timedelta(hours=3):%Hh}  ({len(d):,} bài)", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel(""); ax.set_ylabel("")
    fig.tight_layout(); fig.canvas.draw()
    khung.append(PILImage.fromarray(np.asarray(fig.canvas.buffer_rgba())).convert("P", palette=PILImage.ADAPTIVE))
    plt.close(fig)

khung[0].save("flu_spread.gif", save_all=True, append_images=khung[1:], duration=450, loop=0)
print("Đã lưu flu_spread.gif:", len(khung), "khung hình, mỗi khung 3 giờ")
try:
    from IPython.display import Image as GifImage, display
    display(GifImage("flu_spread.gif"))
except ImportError:
    pass

In [ ]:
d = df[(df["t"] >= "2011-05-18") & (df["t"] < "2011-05-21") & df["flu"]].copy()
huong_ngay = d["ngay"].map(weather.set_index("ngay")["gio_den"])        # hướng gió của chính ngày đó
dx, dy = (d["lon"] - TAM[NGUON][0]) * kx, (d["lat"] - TAM[NGUON][1]) * ky
a = np.radians(huong_ngay)
d["doc"] = dx * np.sin(a) + dy * np.cos(a)                                 # dặm dọc theo hướng gió
d["khung"] = d["t"].dt.floor("6h")

tk = d.groupby("khung")["doc"].agg(trung_vi="median", p90=lambda s: s.quantile(0.9), so_bai="size")
print(tk.round(2).to_string())

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(tk.index, tk["trung_vi"], color=FLU, lw=2.4, marker="o", label="trung vị")
ax.plot(tk.index, tk["p90"], color=NAVY, lw=2.4, marker="s", label="90% ca nằm gần hơn mốc này")
ax.set_ylabel("khoảng cách xuôi gió từ nghi phạm (dặm)"); ax.set_ylim(0)
ax.set_title("Vùng có ca bệnh có mở rộng thêm theo thời gian không?")
ax.legend(frameon=False); ngay_thang(ax, "%d/%m %Hh"); plt.tight_layout()
plt.show()

Điều đáng chú ý: **ngay từ khung đầu tiên, dải bệnh đã dài sẵn** và các ngày sau gần như lặp lại cùng một dải;
vùng bệnh *không* lan dần ra xa nghi phạm. Một nguồn phát tán một lần theo gió sẽ cho hình ảnh như vậy;
một bệnh lây dần từ người sang người thường sẽ cho vòng tròn nở rộng theo thời gian.
Nhưng đây mới là **một** nhóm triệu chứng. Còn nhóm dạ dày - ruột? Nó nằm ở phía tây nam, khá giống một **con sông**.
Ta số hóa đường đi của sông Vast từ ảnh bản đồ, rồi đo khoảng cách từ mỗi bài đăng đến sông.

In [ ]:
SONG = np.array([(-93.350, 42.302), (-93.356, 42.290), (-93.362, 42.275), (-93.367, 42.262), (-93.375, 42.250),
                 (-93.390, 42.241), (-93.405, 42.236), (-93.418, 42.231), (-93.430, 42.226), (-93.445, 42.219),
                 (-93.46, 42.212), (-93.48, 42.204), (-93.50, 42.196), (-93.515, 42.187), (-93.525, 42.178),
                 (-93.530, 42.168), (-93.533, 42.160)])      # sông Vast, số hóa tay từ ảnh bản đồ (bắc xuống nam)
xy = np.c_[SONG[:, 0] * kx, SONG[:, 1] * ky]
doan = np.diff(xy, axis=0)
dai = np.hypot(*doan.T)
cong_don = np.r_[0, dai.cumsum()]


def toa_do_song(lon, lat):
    """Trả về (khoảng cách đến sông, vị trí dọc sông tính từ phía bắc), đơn vị dặm."""
    P = np.c_[np.asarray(lon) * kx, np.asarray(lat) * ky]
    gan, vt = np.full(len(P), 1e9), np.zeros(len(P))
    for a, v, L, c0 in zip(xy[:-1], doan, dai, cong_don[:-1]):
        t = np.clip(((P - a) @ v) / (L * L), 0, 1)
        kc = np.hypot(*(P - (a + t[:, None] * v)).T)
        tot = kc < gan
        gan[tot], vt[tot] = kc[tot], c0 + t[tot] * L
    return gan, vt


gi19 = ngay19[ngay19["gi"]]
kc_gi, _ = toa_do_song(gi19["lon"], gi19["lat"])
kc_all, _ = toa_do_song(ngay19["lon"], ngay19["lat"])
for r in (1.0, 1.5, 3.0):
    print(f"Trong {r} dặm quanh sông: {100 * (kc_gi < r).mean():.1f}% bài dạ dày - ruột, "
          f"{100 * (kc_all < r).mean():.1f}% mọi bài cùng ngày")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.5, 1]})
base_map(axes[0], 0.3)
axes[0].scatter(gi19["lon"], gi19["lat"], s=1.5, c=GI, alpha=0.35)
axes[0].plot(SONG[:, 0], SONG[:, 1], color=NAVY, lw=1.6, ls="--")
axes[0].set_title("Bài dạ dày - ruột (19/5) và đường sông")
axes[1].hist(kc_all, bins=np.arange(0, 8, 0.25), density=True, color=GRAY, alpha=0.6, label="mọi bài")
axes[1].hist(kc_gi, bins=np.arange(0, 8, 0.25), density=True, color=GI, alpha=0.7, label="dạ dày - ruột")
axes[1].axvline(1.5, color=NAVY, ls=":")
axes[1].set_xlabel("khoảng cách đến sông (dặm)"); axes[1].set_ylabel("mật độ"); axes[1].legend(frameon=False)
axes[1].set_title("Khoảng cách đến sông")
plt.tight_layout()
plt.show()

Bản đồ tĩnh chỉ cho ta một ảnh chụp. Để xem bệnh dạ dày - ruột **lan ra theo thời gian**, ta làm ảnh động như với bệnh cúm:
mỗi khung 3 giờ, từ 0h ngày 19/5 đến hết ngày 20/5, kèm đường sông (nét đứt) và nghi phạm số một (ngôi sao vàng).

In [ ]:
khung_gi = []
moc3_gi = pd.date_range("2011-05-19", periods=16, freq="3h")
for t0 in moc3_gi:
    d = df[(df["t"] >= t0) & (df["t"] < t0 + pd.Timedelta(hours=3)) & df["gi"]]
    fig, ax = plt.subplots(figsize=(6.4, 3.4), dpi=80)
    base_map(ax, 0.3)
    ax.scatter(d["lon"], d["lat"], s=1.6, c=GI, alpha=0.45)
    ax.plot(SONG[:, 0], SONG[:, 1], color=NAVY, lw=1.4, ls="--")
    ax.plot(*TAM[NGUON], marker="*", ms=13, mfc=GOLD, mec=NAVY, zorder=5)
    ax.set_title(f"Ca dạ dày - ruột, {t0:%d/%m %Hh} đến {t0 + pd.Timedelta(hours=3):%Hh}  ({len(d):,} bài)", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel(""); ax.set_ylabel("")
    fig.tight_layout(); fig.canvas.draw()
    khung_gi.append(PILImage.fromarray(np.asarray(fig.canvas.buffer_rgba())).convert("P", palette=PILImage.ADAPTIVE))
    plt.close(fig)

khung_gi[0].save("gi_spread.gif", save_all=True, append_images=khung_gi[1:], duration=500, loop=0)
print("Đã lưu gi_spread.gif:", len(khung_gi), "khung hình, mỗi khung 3 giờ")
try:
    display(GifImage("gi_spread.gif"))
except NameError:
    pass

Bài dạ dày - ruột **dồn sát bờ sông**, còn bài thông thường rải đều khắp thành phố.
Một con sông chảy từ đâu đến đâu? Theo README, sông Vast **chảy về phía nam** với tốc độ 3 dặm/giờ.
Nếu nguồn nằm trên sông, ca bệnh phải ở **phía hạ lưu** của nguồn.
Một nguồn gây bệnh qua nước phải **nằm sát sông**, và ca bệnh phải ở **hạ lưu** của nó.
Giờ ta làm phép thử chung cho **cả hai nhóm triệu chứng**: mỗi nghi phạm giải thích được bao nhiêu phần trăm ca cúm
(theo gió) *và* có nằm sát sông không (đo bằng khoảng cách từ nghi phạm đến sông)?

In [ ]:
vt_gi = toa_do_song(gi19["lon"], gi19["lat"])
hang = []
for ten_sk, tam in TAM.items():
    cach_song, vt_nguon = toa_do_song([tam[0]], [tam[1]])
    ha_luu = (vt_gi[1] > vt_nguon[0]) & (vt_gi[0] < 1.5)         # nằm phía hạ lưu và sát sông
    hang.append([ten_sk, round(cach_song[0], 2), 100 * ha_luu.mean()])
song = pd.DataFrame(hang, columns=["sự kiện", "cách sông (dặm)", "% ca dạ dày ở hạ lưu"]).set_index("sự kiện")
tong = diem.join(song)[["% ca trong dải xuôi gió", "cách sông (dặm)", "% ca dạ dày ở hạ lưu"]]
print(tong.round(1).sort_values("% ca trong dải xuôi gió", ascending=False).to_string())

fig, ax = plt.subplots(figsize=(7.5, 5.2))
for ten_sk, r in tong.iterrows():
    tot = ten_sk == NGUON
    ax.scatter(r["% ca trong dải xuôi gió"], r["cách sông (dặm)"], s=240 if tot else 90,
               color=GOLD if tot else GRAY, edgecolor=NAVY, zorder=3)
    ax.annotate(ten_sk, (r["% ca trong dải xuôi gió"], r["cách sông (dặm)"]),
                xytext=(7, -14) if ten_sk == "Hội nghị" else (7, 6),
                textcoords="offset points", fontsize=8, weight="bold" if tot else "normal")
ax.set_xlim(-5, 105); ax.invert_yaxis()
ax.set_xlabel("% ca giống cúm trong dải xuôi gió của sự kiện (càng sang phải càng khớp)")
ax.set_ylabel("khoảng cách từ sự kiện đến sông (dặm, càng lên cao càng gần)")
ax.set_title("Mỗi nghi phạm khớp với hai nhóm triệu chứng đến đâu?")
plt.tight_layout()
plt.show()

Chỉ **một** nghi phạm nằm ở góc trên bên phải: nó vừa **nằm trên sông** vừa giải thích hầu hết ca cúm theo gió.
Các nghi phạm khác hoặc xa sông, hoặc không khớp hướng gió, hoặc ca bệnh nằm *ngược* phía so với nó.
Đây là bằng chứng *hội tụ* từ nhiều nguồn dữ liệu độc lập (vị trí bài đăng, thời tiết, bản đồ). Nhưng cần so sánh ba cơ chế lây lan có thể có:
**không khí**, **nước**, **người sang người**.

In [ ]:
def chi_so(d, huong):
    """Hai chỉ số cho một tập bài: % trong dải xuôi gió của nguồn, và % sát sông (<= 1,5 dặm)."""
    doc, ngang = toa_do_gio(d, TAM[NGUON], huong)
    trong_dai = ((doc > 0) & (doc < 12) & (np.abs(ngang) < 1.5)).mean()
    sat_song = (toa_do_song(d["lon"], d["lat"])[0] < 1.5).mean()
    return [100 * trong_dai, 100 * sat_song]


g = weather.set_index("ngay")["gio_den"]
cum = df[(df["ngay"] == date(2011, 5, 18)) & df["flu"]]
da_day = df[(df["ngay"] == date(2011, 5, 19)) & df["gi"]]
thuong = df[df["ngay"] == date(2011, 5, 19)]

bang_cs = pd.DataFrame([chi_so(cum, g[date(2011, 5, 18)]), chi_so(da_day, g[date(2011, 5, 19)]),
                        chi_so(thuong, g[date(2011, 5, 19)])],
                       index=["ca giống cúm (18/5)", "ca dạ dày - ruột (19/5)", "mọi bài (19/5)"],
                       columns=["% trong dải xuôi gió (không khí)", "% sát sông (nước)"])
print(bang_cs.round(1).to_string())

fig, ax = plt.subplots(figsize=(7.5, 3.2))
im = ax.imshow(bang_cs.values, cmap="Blues", vmin=0, vmax=100, aspect="auto")
ax.set_xticks([0, 1]); ax.set_xticklabels(["xuôi gió\n(không khí)", "sát sông\n(nước)"])
ax.set_yticks(range(3)); ax.set_yticklabels(bang_cs.index)
for i in range(3):
    for j in range(2):
        v = bang_cs.values[i, j]
        ax.text(j, i, f"{v:.0f}%", ha="center", va="center", color="white" if v > 55 else NAVY, weight="bold")
ax.grid(False); ax.set_title("Mỗi nhóm triệu chứng khớp với cơ chế nào?")
plt.tight_layout()
plt.show()

In [ ]:
# Bài "nói về người khác": có tên riêng + has/is, hoặc các đại từ he/she/they...
NGUOI_KHAC = r"\b[A-Z][a-z]{2,} (?:has|is|had) |\b(?:[Tt]hey|[Tt]hem|[Tt]heir|[Ss]he|[Hh]e|[Hh]is|[Hh]er)\b"
df["nguoi_khac"] = df["text"].str.contains(NGUOI_KHAC)
print("Ví dụ bài nói về người khác:")
print(df[df["flu"] & df["nguoi_khac"]]["text"].head(3).to_string(index=False))


def thong_ke_nguoi(cot):
    d = df[df[cot] & (df["t"] >= "2011-05-17")]
    tu_bao = d[~d["nguoi_khac"]]                                    # bài tự kể về mình
    lan_dau = tu_bao.groupby("ID")["ngay"].min().value_counts().sort_index()
    return pd.DataFrame({
        "bài": d.groupby("ngay").size(),
        "% nói về người khác": (100 * d.groupby("ngay")["nguoi_khac"].mean()).round(0),
        "người tự báo": tu_bao.groupby("ngay")["ID"].nunique(),
        "người tự báo mới": lan_dau}).fillna(0).astype(int)


tk_flu, tk_gi = thong_ke_nguoi("flu"), thong_ke_nguoi("gi")
print("Nhóm giống cúm:"); print(tk_flu.to_string())
print("Nhóm dạ dày - ruột:"); print(tk_gi.to_string())

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
x = np.arange(len(tk_flu))
nhan_ngay = [d.strftime("%d/%m") for d in tk_flu.index]
axes[0].bar(x, tk_flu["người tự báo"], color=FLU, alpha=0.35, label="người tự báo (tổng)")
axes[0].bar(x, tk_flu["người tự báo mới"], color=FLU, label="trong đó, người mới")
axes[0].set_xticks(x); axes[0].set_xticklabels(nhan_ngay); axes[0].legend(frameon=False, fontsize=8)
axes[0].set_title("Người tự báo triệu chứng cúm: số người mới mỗi ngày")
axes[1].plot(x, tk_flu["% nói về người khác"], color=FLU, marker="o", lw=2.4, label="giống cúm")
axes[1].plot(tk_gi.index.map(lambda d: (d - tk_flu.index[0]).days), tk_gi["% nói về người khác"],
             color=GI, marker="o", lw=2.4, label="dạ dày - ruột")
axes[1].set_xticks(x); axes[1].set_xticklabels(nhan_ngay); axes[1].legend(frameon=False, fontsize=8)
axes[1].set_ylabel("% bài nói về người khác"); axes[1].set_title("Bài nói về người khác")
plt.tight_layout()
plt.show()

Nếu bệnh lây từ người sang người, số **người mới** tự báo bệnh mỗi ngày thường *tăng dần theo cấp số nhân*.
Ở đây số người tự báo mới **sụt xuống rất nhanh** sau ngày đầu, và phần lớn người bệnh **đăng lại** những ngày sau.
Trong khi đó, tỉ lệ bài *nói về người khác* lại tăng: lượng bài đăng tăng chủ yếu vì **nỗi lo và tin đồn**,
không phải vì có thêm nhiều người bệnh. Điều này *không hỗ trợ* giả thuyết lây theo cấp số nhân,
nhưng cũng **chưa thể loại trừ** hoàn toàn: bài đăng không phải xét nghiệm y tế, và cách nhận diện bài nói về người khác
chỉ là một quy tắc đoán.

## Bước 8. Hệ thống y tế và hành động

Ngày 20/5 người bệnh bắt đầu tìm đến bệnh viện. Nếu đúng, các bài giống cúm của ngày này phải **dồn thành từng cụm nhỏ**.
Dùng thuật toán gom cụm theo mật độ (DBSCAN) để tìm các cụm đó, rồi đặt lên bản đồ.

In [ ]:
from sklearn.cluster import DBSCAN

sang20 = df[(df["t"] >= "2011-05-20 08:00") & df["flu"]]
P = np.c_[sang20["lon"] * kx, sang20["lat"] * ky]
nhan_cum = DBSCAN(eps=0.15, min_samples=80).fit_predict(P)              # cụm: ít nhất 80 bài trong bán kính 0,15 dặm
cum_tam = np.array([[sang20["lon"][nhan_cum == k].mean(), sang20["lat"][nhan_cum == k].mean()]
                    for k in range(nhan_cum.max() + 1)])
print(f"DBSCAN tìm thấy {len(cum_tam)} cụm, chiếm {100 * (nhan_cum >= 0).mean():.0f}% bài giống cúm ngày 20/5")

BENH_VIEN = {   # tên bệnh viện (theo README) và tọa độ biểu tượng trên ảnh bản đồ
    "Corner": (-93.5317, 42.2987), "Westside": (-93.4788, 42.2565), "Northside": (-93.42, 42.2526),
    "Villa": (-93.5567, 42.2043), "Smogtown": (-93.4947, 42.2019), "Plainville": (-93.4451, 42.2048),
    "Southville": (-93.3908, 42.174), "Vastopolis City": (-93.3622, 42.2201), "St. Georges": (-93.3303, 42.2404),
    "Lakeside": (-93.3432, 42.1682), "St. Kristin's": (-93.2469, 42.2197), "St. Mark's": (-93.2339, 42.2947),
    "River": (-93.3649, 42.2844),
}
bv = np.array(list(BENH_VIEN.values()))
for k, c in enumerate(cum_tam):
    kc = np.hypot((bv[:, 0] - c[0]) * kx, (bv[:, 1] - c[1]) * ky)
    ten_bv = list(BENH_VIEN)[kc.argmin()] if kc.min() < 0.3 else "không phải bệnh viện"
    print(f"cụm {k:2d}: {np.round(c, 4)}  {(nhan_cum == k).sum():4d} bài  ->  {ten_bv}")

fig, ax = plt.subplots(figsize=(10, 5.2))
base_map(ax, 0.4)
ax.scatter(sang20["lon"], sang20["lat"], s=1, c=FLU, alpha=0.25)
ax.scatter(cum_tam[:, 0], cum_tam[:, 1], s=260, facecolor="none", edgecolor=NAVY, linewidth=2, zorder=4, label="cụm DBSCAN")
ax.scatter(bv[:, 0], bv[:, 1], marker="P", s=70, color=GOLD, edgecolor=NAVY, zorder=5, label="bệnh viện")
ax.set_title("Bài giống cúm từ 8h ngày 20/5: các cụm trùng với bệnh viện")
ax.legend(frameon=True, loc="lower right"); plt.tight_layout()
plt.show()

In [ ]:
def gan_benh_vien(d, r=0.35):
    """Tỉ lệ bài nằm trong r dặm quanh một bệnh viện."""
    kc = np.hypot((d["lon"].values[:, None] - bv[:, 0]) * kx, (d["lat"].values[:, None] - bv[:, 1]) * ky)
    return (kc.min(1) < r).mean()


df["flu_tu"] = df["flu"] & ~df["nguoi_khac"]               # bài cúm do người bệnh tự kể

hang = []
for ngay in [date(2011, 5, 17), date(2011, 5, 18), date(2011, 5, 19), date(2011, 5, 20)]:
    ban_ngay_ = df[(df["ngay"] == ngay) & (df["t"].dt.hour >= 8)]
    hang.append([ngay.strftime("%d/%m"), 100 * gan_benh_vien(ban_ngay_[ban_ngay_["flu_tu"]]),
                 100 * gan_benh_vien(ban_ngay_[ban_ngay_["gi"]]), 100 * gan_benh_vien(ban_ngay_[~ban_ngay_["flu"]])])
bang_bv = pd.DataFrame(hang, columns=["ngày", "% bài cúm tự báo gần BV", "% bài dạ dày gần BV", "% bài khác gần BV"])
print(bang_bv.round(1).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 3.4))
x = np.arange(len(bang_bv)); rong = 0.27
ax.bar(x - rong, bang_bv.iloc[:, 1], rong, color=FLU, label="giống cúm (tự báo)")
ax.bar(x, bang_bv.iloc[:, 2], rong, color=GI, label="dạ dày - ruột")
ax.bar(x + rong, bang_bv.iloc[:, 3], rong, color=GRAY, label="bài khác")
ax.set_xticks(x); ax.set_xticklabels(bang_bv["ngày"])
ax.set_ylabel("% bài trong 0,35 dặm quanh bệnh viện"); ax.legend(frameon=False)
ax.set_title("Người bệnh dồn về bệnh viện vào ngày nào?")
plt.tight_layout()
plt.show()

Còn việc cuối: **khoanh vùng hành động**. Nếu nguồn lực có hạn, nên tập trung vào đâu?
Ta tìm tập ô lưới nhỏ nhất chứa **80%** số bài triệu chứng vượt mức bình thường của ngày 19/5.

In [ ]:
tc19 = ngay19[ngay19["flu"] | ngay19["gi"]]
tc_nen = nen[nen["flu"] | nen["gi"]]
vuot = np.clip(luoi(tc19) - luoi(tc_nen) / n_nen, 0, None)
vuot = gaussian_filter(vuot, 1.0)

thu_tu = np.sort(vuot.ravel())[::-1]
so_o = np.searchsorted(np.cumsum(thu_tu) / thu_tu.sum(), 0.8) + 1
nguong = thu_tu[so_o - 1]
dien_tich_o = (xb[1] - xb[0]) * kx * (yb[1] - yb[0]) * ky
tong_dt = (EXTENT[1] - EXTENT[0]) * kx * (EXTENT[3] - EXTENT[2]) * ky
print(f"Vùng ưu tiên chứa 80% bài vượt mức: {so_o} ô, khoảng {so_o * dien_tich_o:.0f} dặm vuông, "
      f"bằng {100 * so_o * dien_tich_o / tong_dt:.0f}% diện tích thành phố ({tong_dt:.0f} dặm vuông)")

fig, ax = plt.subplots(figsize=(10, 5.2))
base_map(ax, 0.4)
ax.contourf((xb[:-1] + xb[1:]) / 2, (yb[:-1] + yb[1:]) / 2, vuot, levels=[nguong, vuot.max() * 2],
            colors=[FLU], alpha=0.45, zorder=2)
ax.plot(SONG[:, 0], SONG[:, 1], color=NAVY, lw=1.6, ls="--", zorder=3)
ax.plot(*TAM[NGUON], marker="*", ms=16, mfc=GOLD, mec=NAVY, zorder=5)
ax.scatter(bv[:, 0], bv[:, 1], marker="P", s=70, color=GOLD, edgecolor=NAVY, zorder=5)
ax.set_title("Vùng ưu tiên (chứa 80% bài triệu chứng vượt mức), sông, bệnh viện")
plt.tight_layout()
plt.show()

In [ ]:
t_sk = BAI_SK[NGUON]["t"].quantile(0.1)
t_cum = thoi_diem_bat_dau("flu")[0]
gio_flu = df.groupby(df["t"].dt.floor("h"))["flu"].sum()
t_bung = gio_flu[gio_flu > 20 * gio_flu.iloc[: 14 * 24].mean()].index[0]
t_gi = thoi_diem_bat_dau("gi")[0]
hai_20 = df[(df["t"] >= "2011-05-19") & df["flu_tu"]]
ti_le_bv = hai_20.groupby(hai_20["t"].dt.floor("h")).apply(gan_benh_vien)
t_bv = ti_le_bv[ti_le_bv > 0.3].index[0]

cac_moc = [(t_sk, f"{NGUON}\n(10% bài đăng đầu)", EVT), (t_cum, "bắt đầu có\nca giống cúm", FLU),
           (t_bung, "ca giống cúm\nbùng nổ", FLU), (t_gi, "bắt đầu có ca\ndạ dày - ruột", GI),
           (t_bv, "người bệnh dồn\nvề bệnh viện", GOLD)]
for t, ten_moc, _ in cac_moc:
    print(f"{t:%d/%m %H:%M}  {ten_moc.replace(chr(10), ' ')}")

fig, ax = plt.subplots(figsize=(11, 2.8))
ax.hlines(0, cac_moc[0][0] - pd.Timedelta(hours=8), cac_moc[-1][0] + pd.Timedelta(hours=8), color=NAVY, lw=2)
for i, (t, ten_moc, mau) in enumerate(cac_moc):
    ax.scatter([t], [0], s=150, color=mau, edgecolor=NAVY, zorder=3)
    ax.annotate(f"{t:%d/%m %Hh}\n{ten_moc}", (t, 0), xytext=(0, 20 if i % 2 == 0 else -50),
                textcoords="offset points", ha="center", fontsize=8.5)
ax.set_ylim(-1, 1); ax.axis("off"); ax.set_title("Dòng thời gian của vụ dịch (rút ra từ dữ liệu)")
plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(12, 5.2))
gs = fig.add_gridspec(2, 2, width_ratios=[1.8, 1])
ax = fig.add_subplot(gs[:, 0])
base_map(ax, 0.35)
mau_flu = df[(df["t"] >= "2011-05-18") & (df["t"] < "2011-05-21") & df["flu"]].sample(9000, random_state=2)
mau_gi = df[(df["t"] >= "2011-05-19") & (df["t"] < "2011-05-21") & df["gi"]].sample(5000, random_state=2)
ax.scatter(mau_flu["lon"], mau_flu["lat"], s=1.2, c=FLU, alpha=0.3, label="ca giống cúm")
ax.scatter(mau_gi["lon"], mau_gi["lat"], s=1.2, c=GI, alpha=0.3, label="ca dạ dày - ruột")
ax.plot(SONG[:, 0], SONG[:, 1], color=NAVY, lw=1.4, ls="--")
ax.scatter(bv[:, 0], bv[:, 1], marker="P", s=60, color=GOLD, edgecolor=NAVY, zorder=5, label="bệnh viện")
ax.plot(*TAM[NGUON], marker="*", ms=16, mfc=GOLD, mec=NAVY, zorder=6, label=NGUON)
ax.legend(loc="lower right", fontsize=8, markerscale=4); ax.set_title("Bản đồ tổng hợp")
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(theo_ngay.index, theo_ngay["flu"], color=FLU, lw=2); ax2.plot(theo_ngay.index, theo_ngay["gi"], color=GI, lw=2)
ngay_thang(ax2); ax2.set_title("Bài triệu chứng mỗi ngày", fontsize=10); ax2.tick_params(labelsize=7)
ax3 = fig.add_subplot(gs[1, 1])
ax3.bar(range(4), bang_bv.iloc[:, 1], color=FLU); ax3.set_xticks(range(4)); ax3.set_xticklabels(bang_bv["ngày"])
ax3.set_title("% bài cúm gần bệnh viện", fontsize=10); ax3.tick_params(labelsize=7)
plt.tight_layout()
plt.show()

## Kết luận và lưu ý

Từ dữ liệu, ta rút ra các **giả thuyết được dữ liệu hỗ trợ** (không phải kết quả xét nghiệm):

1. Hai nhóm triệu chứng có hai dạng phân bố khác nhau: một nhóm theo **hướng gió**, một nhóm theo **dòng sông**.
2. Một sự kiện duy nhất (nghi phạm số một) giải thích được cả hai nhóm.
3. Số người mới mỗi ngày không tăng theo cấp số nhân, nên **không có bằng chứng cho lây từ người sang người** trong dữ liệu này.
4. Từ 20/5, người bệnh dồn về các bệnh viện; cần tập trung nguồn lực vào vùng ưu tiên và các bệnh viện.

Lưu ý khi kể chuyện với dữ liệu:

- **Tương quan không phải nhân quả.** Ta chỉ có vị trí và thời gian của *bài đăng*, không có xét nghiệm.
- **Bài đăng không phải ca bệnh.** Một người có thể đăng nhiều lần, và có người đăng về người khác.
- **Vị trí bài đăng không chắc là nơi bị nhiễm.** Người ta đi làm, di chuyển.
- Đây là **dữ liệu mô phỏng** do ban tổ chức tạo ra, nên con số không phản ánh bệnh thật.
- Các nhóm tham gia VAST Challenge 2011 không hoàn toàn thống nhất (ví dụ về mức độ kiểm soát dịch và khả năng
  lây từ người sang người); kết luận trên chỉ là cách đọc của riêng bài này.

### Đối chiếu với đáp án chính thức

Ban tổ chức công bố *Answer Guide* của Mini-Challenge 1 (xem phần nguồn). Hãy mở nó và so với kết quả của bạn:
tai nạn xe tải trên cầu I-610 ngày 17/5 (42,22655 độ Bắc, 93,42752 độ Tây), gió Tây thổi sang Đông,
ca giống cúm lan theo gió từ 18/5, ca dạ dày - ruột đi theo dòng sông từ 19/5, người bệnh chọn bệnh viện ngẫu nhiên từ 20/5.

### Nguồn

- VAST Challenge 2011, Mini-Challenge 1: Characterization of an Epidemic Spread (Visual Analytics Benchmark Repository, HCIL, ĐH Maryland).
- Answer Guide MC1 (tệp `MC1.pdf` trong kho lưu trữ trên).